In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import train_test_split, cross_validate

from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

In [2]:
DATA_DIR = Path("../data/raw")

train = pd.read_csv(DATA_DIR / "train.csv")

target = "SalePrice"
id_col = "Id"

y = train[target]
X = train.drop(columns=[target, id_col])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1460, 79)
y shape: (1460,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1168, 79)
X_test: (292, 79)
y_train: (1168,)
y_test: (292,)


In [5]:
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 36
Categorical features: 43


C:\Users\Александр\AppData\Local\Temp\ipykernel_39004\3208774431.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()


Блок 5 — preprocessors

Для линейных моделей оставляем scaler. Для деревьев scaler не нужен, но для честной простоты можно использовать один общий preprocessing. Это не leakage, потому что scaler внутри CV.

In [10]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

categorical_pipeline_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_dense = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline_dense, categorical_features)
])

Блок 6 — metrics

In [11]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5

scoring = {
    "mae": make_scorer(mean_absolute_error, greater_is_better=False),
    "rmse": make_scorer(rmse, greater_is_better=False),
    "r2": make_scorer(r2_score)
}

Блок 7 — models

In [ ]:
models = {
    "DummyRegressor_mean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "DecisionTreeRegressor_max_depth_3": DecisionTreeRegressor(
        max_depth=3,
        random_state=42
    ),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "ExtraTreesRegressor": ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(
        random_state=42
    ),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(
        random_state=42
    ),
}

Блок 8 — CV on X_train only

In [12]:
results = []

for model_name, model in models.items():
    
    if model_name == "HistGradientBoostingRegressor":
        preprocessor = preprocessor_dense  
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    cv_results = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring=scoring,
        return_train_score=False,
        n_jobs=-1
    )
    
    results.append({
        "model": model_name,
        "MAE_mean": -cv_results["test_mae"].mean(),
        "MAE_std": cv_results["test_mae"].std(),
        "RMSE_mean": -cv_results["test_rmse"].mean(),
        "RMSE_std": cv_results["test_rmse"].std(),
        "R2_mean": cv_results["test_r2"].mean(),
        "R2_std": cv_results["test_r2"].std(),
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("RMSE_mean")
    .reset_index(drop=True)
)

display(results_df)

,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
0,HistGradientBoostingRegressor,17279.885742,1823.199437,29404.090922,5353.785125,0.851486,0.045032
1,GradientBoostingRegressor,16686.977286,1403.319792,29571.126861,4341.926857,0.850062,0.039341
2,RandomForestRegressor,18198.367144,1637.266973,30583.721919,4855.965157,0.838639,0.048042
3,ExtraTreesRegressor,19316.478522,1945.930518,32843.596173,5995.794061,0.817077,0.044343
4,Ridge,18758.161542,1121.088500,33857.889344,8060.243798,0.802895,0.073313
5,LinearRegression,19791.303140,1464.107683,43613.528886,16218.650620,0.654856,0.224416
6,DecisionTreeRegressor_max_depth_3,33012.263973,2142.582715,48997.246238,6126.575745,0.592987,0.064216
7,DummyRegressor_mean,56340.483620,2891.420120,77051.137457,5826.958206,-0.003663,0.003976


Блок 9 — compare against Ridge

In [15]:
ridge_rmse = results_df.loc[results_df["model"] == "Ridge", "RMSE_mean"].iloc[0]

results_df["RMSE_vs_Ridge"] = results_df["RMSE_mean"] - ridge_rmse
results_df["RMSE_improvement_vs_Ridge"] = ridge_rmse - results_df["RMSE_mean"]

display(results_df.sort_values("RMSE_mean"))

,model,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std,RMSE_vs_Ridge,RMSE_improvement_vs_Ridge
0,HistGradientBoostingRegressor,17279.885742,1823.199437,29404.090922,5353.785125,0.851486,0.045032,-4453.798422,4453.798422
1,GradientBoostingRegressor,16686.977286,1403.319792,29571.126861,4341.926857,0.850062,0.039341,-4286.762483,4286.762483
2,RandomForestRegressor,18198.367144,1637.266973,30583.721919,4855.965157,0.838639,0.048042,-3274.167426,3274.167426
3,ExtraTreesRegressor,19316.478522,1945.930518,32843.596173,5995.794061,0.817077,0.044343,-1014.293171,1014.293171
4,Ridge,18758.161542,1121.088500,33857.889344,8060.243798,0.802895,0.073313,0.000000,0.000000
5,LinearRegression,19791.303140,1464.107683,43613.528886,16218.650620,0.654856,0.224416,9755.639542,-9755.639542
6,DecisionTreeRegressor_max_depth_3,33012.263973,2142.582715,48997.246238,6126.575745,0.592987,0.064216,15139.356893,-15139.356893
7,DummyRegressor_mean,56340.483620,2891.420120,77051.137457,5826.958206,-0.003663,0.003976,43193.248113,-43193.248113


## Stage 4 model comparison conclusions

### Setup
- Same data preparation as Stage 3.
- Target: SalePrice.
- X excludes SalePrice and Id.
- Same local split: test_size=0.2, random_state=42.
- Local X_test/y_test were not evaluated.
- Official Kaggle test.csv was not used.
- CV was performed only on X_train.

### Preprocessing
- Numeric features: SimpleImputer(strategy="median") + StandardScaler.
- Categorical features: SimpleImputer(strategy="constant", fill_value="None") + OneHotEncoder(handle_unknown="ignore").
- All preprocessing was inside Pipeline / ColumnTransformer.

### Models compared
- DummyRegressor
- LinearRegression
- Ridge
- DecisionTreeRegressor(max_depth=3)
- RandomForestRegressor
- ExtraTreesRegressor
- GradientBoostingRegressor
- HistGradientBoostingRegressor

### Main findings
- Ridge remains the reference baseline from Stage 3.
- Ensemble/tree/boosting models are compared against Ridge by CV RMSE/MAE.
- No model was tuned.
- No outliers were removed.
- No log target transformation was used in the main comparison.

### Later candidates
- Select 2–3 strongest candidates for future hyperparameter tuning and/or log-target experiments.